# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanniyaAslam/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/SanniyaAslam/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Working dir:", os.getcwd())

Connected. Working dir: /content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*One row = one client, one content item, on one report date — the grain of
fact_content_daily_performance (client_hash_id × content_hash_id × report_date).
I'm joining this with dim_content (content_hash_id) for content attributes like
word_count and content_type, and dim_clients (client_hash_id) to confirm data
availability per client.

Time window: month=2026-03 (mid-panel, for building/iterating). The final month
(2026-06) is a sealed test month — never used to develop label logic, since it's
the natural outcome window for any past→future trend label.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_grain_keys,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys,min_date,max_date
0,9841378,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*Feature: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_engaged_sessions,
sessions_organic, word_count, content_type — all knowable from observed daily
performance and static content attributes.

Label / proxy: a content item is "declining" if its clicks in the second half
of March 2026 are lower than the first half — a within-month trend proxy,
computed only from March data (no peeking into later months).

Context (not a feature, used for filtering/grouping): client_hash_id,
is_active, has_gsc_access, has_ga4_access, access_profile — used to confirm
which rows have real, available data before trusting them.

Excluded: last_optimized_date and optimization_eligible_date. These reflect
a decision someone already made about the content, so using them to predict
"should this be refreshed" would be leakage — the feature would encode the
answer. Also excluded: any month other than 2026-03 for building the label,
since a later month is future information relative to the decision point.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.print("Feature / label / context / excluded — documented above. No query needed for this section.")
print("Grain check: see Section 1 — total_rows == distinct_grain_keys confirms one row per client+content+date.")



Grain check: see Section 1 — total_rows == distinct_grain_keys confirms one row per client+content+date.


In [13]:
q3b = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
q3b

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [14]:
q3c = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
q3c

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061.0,413966.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*This data can never tell me: whether refreshing a page causes recovery
(no controlled experiment exists here — only observed correlation). It also
can't be trusted equally across clients: gsc_data_start and ga4_data_start
in dim_clients show this is an unbalanced panel — some clients have far more
history than others, and some rows may be GSC-only or GA4-only depending on
client access (has_gsc_access / has_ga4_access). A within-month trend label
(first half vs second half of March) is a short window and may be noisy —
a longer or different window could shift results. Finally, this is
pseudonymized data: no client, URL, or keyword can be identified, so any
finding is directional, not attributable to a specific real client.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Limits documented above — no query needed for this section.")

Limits documented above — no query needed for this section.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.